# 05. Ngẫu nhiên hóa

Kích thước lô, bước khởi đầu, và bốn quy tắc giảm bước. Chương 6 của báo cáo.

Lưu ý về tốc độ giảm: $\gamma$ phải neo vào $\mu$ chứ không vào số vòng mỗi epoch.
Hai giá trị hợp lý cho hai kết luận ngược nhau.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_processed, SWEEP
from src.objective import max_row_smoothness, batch_smoothness
from src.experiment import BUILDERS, ExperimentGroup, run_group, seed_band
from src.figures import convergence_pair, band_figure, save_figure
obj, *_ = load_processed("../data/processed", SWEEP)
print(f"L = {obj.L:.4f}   L_max = {max_row_smoothness(obj):.1f}   ty le = {max_row_smoothness(obj)/obj.L:.0f}x")
for B in (32, 256, 2048):
    print(f"  B={B:>5}  L_B={batch_smoothness(obj,B):9.2f}  1/L_B={1/batch_smoothness(obj,B):.6f}")

L = 9.1156   L_max = 200055.9   ty le = 21947x
  B=   32  L_B=  6260.58  1/L_B=0.000160
  B=  256  L_B=   790.55  1/L_B=0.001265
  B= 2048  L_B=   106.79  1/L_B=0.009364


In [3]:
groups = {}
for name in ("batch-size", "batch-eta", "batch-schedule"):
    title, build = BUILDERS[name]
    groups[name] = run_group(ExperimentGroup(name, title, build), obj, "../results/raw")

batch-size: loaded 8 runs from ../results/raw/batch-size.json
batch-eta: loaded 8 runs from ../results/raw/batch-eta.json
batch-schedule: loaded 25 runs from ../results/raw/batch-schedule.json


In [4]:
for name, recs in groups.items():
    show = [r for r in recs if r.meta.get("seed", 0) == 0]
    convergence_pair(show, name, title=BUILDERS[name][0],
                     axes=("iters", "time", "epochs"), out_dir="../results/figures")
bands = seed_band(groups["batch-schedule"], "schedule")
save_figure(band_figure(bands, axis="epochs", title="Step-size schedules, 5 seeds"),
            "batch-schedule_band", "../results/figures")
{k: v["n_seeds"] for k, v in bands.items()}

{'constant': 5,
 'inverse (gamma = mu*eta0)': 5,
 'inverse (gamma = 1/epoch)': 5,
 'sqrt': 5,
 'staircase': 5}

In [5]:
pd.DataFrame([{"quy tắc": k, "trung vị cuối": v["median"][-1],
               "thấp nhất": v["low"][-1], "cao nhất": v["high"][-1]}
              for k, v in bands.items()])

,quy tắc,trung vị cuối,thấp nhất,cao nhất
0,constant,3.032753e-04,1.318706e-04,0.001056
1,inverse (gamma = mu*eta0),5.851022e-06,3.828808e-06,0.000007
2,inverse (gamma = 1/epoch),1.461315e-02,1.458826e-02,0.014626
3,sqrt,4.582021e-01,4.578907e-01,0.458400
4,staircase,8.493894e-07,7.134346e-07,0.000001
